# 02 · Drive styles

Cluster drives on **how they were played** (the 11 `STYLE_FEATURES`), never on how they ended.
Pipeline (`drivedna.models.cluster`): impute → shrink short drives toward the league mean → standardize →
PCA (90% variance) → KMeans. This notebook shows the evidence for each choice, then profiles the styles.

*Anything about outcomes below is uncontrolled and descriptive. Phase 4 handles confounding.*

In [ ]:
import numpy as np, pandas as pd, polars as pl
import plotly.express as px, plotly.graph_objects as go
import json, joblib

from drivedna.drives.build import DRIVE_KEY, PROCESSED_DIR
from drivedna.features.drive_features import STYLE_FEATURES, RHYTHM_FEATURES
from drivedna.features.tokens import drive_token_sequences
from drivedna.models.cluster import REPORTS_DIR, STYLE_ORDER, eligible_drives

feats = pl.read_parquet(PROCESSED_DIR / "drive_features.parquet")
styles = pl.read_parquet(PROCESSED_DIR / "drive_styles.parquet")
d = eligible_drives(feats).join(styles, on=DRIVE_KEY)
model = joblib.load(PROCESSED_DIR / "style_model.joblib")
print(f"{d.height:,} eligible drives ({d.height / feats.filter(pl.col('keep')).height:.0%} of kept drives; >=3 designed plays)")
d.group_by('style').len().sort('len', descending=True).with_columns(share=pl.col('len')/pl.col('len').sum())

## 1 · Is there natural cluster structure? (Mostly no.)
Silhouette is low for every k (≤ ~0.17) and BIC for a full-covariance GMM never levels off. Drive styles are a
**continuum**; clusters are a useful partition of it, not discovered species.

In [ ]:
ks = pd.read_csv(REPORTS_DIR / "k_selection.csv")
fig = px.line(ks, x="k", y="silhouette", color="method", markers=True, title="Silhouette by k (higher = better separated)")
fig.show()
px.line(ks, x="k", y="bootstrap_ari", color="method", markers=True,
        title="Bootstrap ARI by k with preprocessing FIXED (weak test: GMM-tied is much less stable than KMeans)").show()

## 2 · Choosing k with the demanding stability test
`k_selection` above fixes preprocessing and is a noisy diagnostic. The real test refits the **whole pipeline** on 30 bootstrap
resamples (comparing each to the full-data model) and checks that a model fit on 2016-2020 agrees with one fit on 2021-2025.
`bootstrap_frac_below_0_8` is how often a refit lands on a *different* partition, which KMeans does when two solutions are nearly tied.

In [ ]:
ksb = pd.read_csv(REPORTS_DIR / "k_stability.csv")
print(ksb.round(3).to_string(index=False))
m = ksb.melt(id_vars="k", value_vars=["bootstrap_ari_mean", "bootstrap_ari_min", "temporal_ari"])
fig = px.line(m, x="k", y="value", color="variable", markers=True, title="Whole-pipeline stability by k (ARI; >0.8 is solid)")
fig.add_hline(y=0.8, line_dash="dot")
fig.show()

## 3 · What the styles look like

In [ ]:
prof = pd.read_csv(REPORTS_DIR / "style_profiles.csv").set_index("style_id")
prof.index = prof["style"]
show = ["share", "n_real_plays"] + STYLE_FEATURES
print(prof[show].round(2).T.to_string())

In [ ]:
# z-score of each style's mean vs. the league (all eligible drives), by feature
league = d.select(STYLE_FEATURES).to_pandas()
z = (prof[STYLE_FEATURES] - league.mean()) / league.std()
fig = px.imshow(z.T, color_continuous_scale="RdBu_r", zmin=-2, zmax=2, aspect="auto", text_auto=".1f",
                title="Style centroids, standard deviations from the league mean (drive level)")
fig.update_layout(height=520)
fig.show()

In [ ]:
fig = go.Figure()
zr = z.clip(-2, 2)
for s in zr.index:
    fig.add_trace(go.Scatterpolar(r=zr.loc[s].tolist() + [zr.loc[s].iloc[0]], theta=STYLE_FEATURES + [STYLE_FEATURES[0]], name=s, fill="toself", opacity=0.45))
fig.update_layout(title="Style radar (z-scores, clipped to ±2)", height=600)
fig.show()

## 4 · Are styles just game situation or drive length in disguise?
Styles are defined by play-calling, so they *should* correlate with situation somewhat (trailing teams hurry up).
What matters is that they are not **only** that, and not an artifact of how many plays the drive had.

In [ ]:
sit = d.group_by("style").agg(
    n_real_plays=pl.col("n_real_plays").mean(),
    score_diff_start=pl.col("score_diff_start").mean(),
    start_yardline_100=pl.col("start_yardline_100").mean(),
    pct_trailing=(pl.col("score_diff_start") < 0).mean(),
    pct_q4=(pl.col("qtr_start") == 4).mean(),
).sort("score_diff_start")
print(sit.to_pandas().round(2).to_string(index=False))
print("\nRange of mean drive length across styles:", round(sit['n_real_plays'].max() - sit['n_real_plays'].min(), 2), "plays")

### Two checks worth seeing
1. **`scramble_rate` is deliberately *not* a clustering feature.** An earlier version included it and one cluster turned out to be mostly
   "this drive had a QB scramble" (83% of its drives had exactly one). Scrambles are partly reactive (pressure, coverage) and gain yards,
   so they are tracked as a descriptive RHYTHM feature instead. Below: no style is defined by scrambles now.
2. **Assignments are soft, and short drives are thin evidence.** Most drives sit fairly close to a boundary between two styles
   (a continuum), and style on a 3-play drive rests on very little.

In [ ]:
print(d.group_by("style").agg(scramble_rate=pl.col("scramble_rate").mean()).sort("style"))

# how much of the variation in drive length is explained by style? (eta squared)
y = d["n_real_plays"].to_numpy().astype(float)
gm = d.group_by("style").agg(m=pl.col("n_real_plays").mean(), n=pl.len())
eta2 = sum(r["n"] * (r["m"] - y.mean()) ** 2 for r in gm.iter_rows(named=True)) / ((y - y.mean()) ** 2).sum()
print()
print(f"Share of drive-length variance explained by style: {eta2:.1%}")

print()
print("Share of each style's drives that are 3 plays long, and how close drives sit to a style boundary (margin = 0 on the boundary):")
print(d.group_by("style").agg(three_play=(pl.col("n_real_plays") == 3).mean(), mean_margin=pl.col("margin").mean(),
                              share_margin_below_0_2=(pl.col("margin") < 0.2).mean()).sort("style"))

## 5 · Example drives per style
Play-call token sequences (`R_in`/`R_out` runs, `P_short`/`P_mid`/`P_deep` passes, `SCR` scramble, `SACK`).

In [ ]:
seq = drive_token_sequences(pl.read_parquet(PROCESSED_DIR / "drive_plays.parquet"))
ex = d.join(seq, on=DRIVE_KEY).filter(pl.col("margin") > 0.3)
for s in STYLE_ORDER:
    print(f"--- {s}")
    for r in ex.filter(pl.col("style") == s).sample(4, seed=3).iter_rows(named=True):
        print(f"  {r['season']} {r['posteam']:>3} ({r['result']:<17}) {r['token_str']}")

## 6 · A map of the drives (UMAP of the model's PCA space)

In [ ]:
import umap
sample = d.sample(12000, seed=0)
emb = umap.UMAP(n_neighbors=40, min_dist=0.3, random_state=0).fit_transform(model.transform(sample))
mp = pd.DataFrame({"x": emb[:, 0], "y": emb[:, 1], "style": sample["style"].to_list(), "margin": sample["margin"].to_list()})
px.scatter(mp, x="x", y="y", color="style", opacity=0.6, title="UMAP of drives colored by style (12k sample)", height=650).show()

## 7 · Do the styles hold across seasons? (and a first look at who runs them)

In [ ]:
mix = d.group_by("season", "style").len().with_columns(share=pl.col("len") / pl.col("len").sum().over("season")).sort("season")
px.bar(mix.to_pandas(), x="season", y="share", color="style", title="Style mix by season").show()

## 8 · Uncontrolled outcomes by style (for orientation only)
**Do not read these as "which style works".** Styles differ in field position, score and opponent (section 4).
Phase 4 controls for all of that.

In [ ]:
out = d.group_by("style").agg(
    share=pl.len() / d.height, points=pl.col("points").mean(), td=(pl.col("result") == "Touchdown").mean(),
    punt=(pl.col("result") == "Punt").mean(), turnover=pl.col("result").is_in(["Turnover", "Turnover on downs"]).mean(),
    epa=pl.col("epa").mean(), start_yardline_100=pl.col("start_yardline_100").mean(),
).sort("points", descending=True)
print(out.to_pandas().round(3).to_string(index=False))

## Takeaways to carry forward

**What we found**
- **Styles are a continuum, not natural clumps.** Silhouette is ≤ ~0.18 for every k, and 56-68% of drives sit within 0.2 margin of a boundary between two styles. The five styles are a *useful partition*, not discovered species.
- **k=5 is the finest partition that is still reproducible, but not bulletproof.** Refitting the whole pipeline on 30 bootstrap resamples gives mean ARI 0.95, but ~1 in 15 refits (7%) lands on a different near-equal partition. A model fit on 2016-2020 agrees with one fit on 2021-2025 on 96% of late-era drives (ARI 0.91). k=3-4 are rock solid (ARI ~0.97, no alternative partitions), so **k=4 is the conservative choice**; k ≥ 6 degrades (temporal ARI 0.53 at k=6).
- **The five styles:** *Quick-game* (68% passing, 66% of passes short; 35% of drives), *Ground-based* (35% passing; 22%), *Vertical* (17 air yards, 55% of passes deep; 20%), *Hurry-up* (92% passing, ~16 s/play, a two-minute-drill style; 16%), *No-huddle* (52% of plays no-huddle, ~28 s/play; 7%).
- **Eras barely differ.** The style mix shifts only slightly from 2016-2020 to 2021-2025 (Vertical ~21.0% → 19.0%, the others within 1 point).
- **Situation shapes style, as expected, but style is not drive length.** No-huddle and Hurry-up drives start with the offense trailing by ~2.5-2.8 points (55-57% trailing) vs. -0.1 for Ground-based (37%). Style explains only 2.3% of the variance in drive length.

**Design decision: `scramble_rate` is not a clustering feature.** With it, one cluster was mostly "this drive had a QB scramble" (83% of its drives had exactly one). Scrambles are partly a reaction to pressure and gain yards, so it is now a descriptive RHYTHM feature, and no style is defined by scrambling (scramble rate is 2-3% in four styles and 1.7% in the fifth).

**Caveats that matter for Phase 4**
- **Style on short drives is thin evidence.** 3-play drives are 32% of Hurry-up and Quick-game but only 15% of No-huddle, and a failed 3-play drive is mostly a few short passes. The uncontrolled outcome table in section 8 (No-huddle 2.5 pts/drive vs. Hurry-up 1.7) mixes style with game script and drive length, so **do not read it as "which style works."**
- **Plan for Phase 4:** control for score, field position and opponent; use `margin` (sensitivity analysis on confident assignments); and consider assigning style from only the *first N plays* so style is not decided by how the drive ended.
